# 🚀 HUẤN LUYỆN & THỰC NGHIỆM MỞ RỘNG SEQ2SEQ LSTM + ATTENTION
**Đề tài 8: Dịch máy Anh - Việt | Môn học: Học sâu**  
**Phụ trách: Hoàng (Phần 2)**  
---
> **Lưu ý:** Chọn GPU trên Colab: **Runtime -> Change runtime type -> T4 GPU**.

### Bước 1: Kiểm tra GPU & Kết nối Google Drive lưu Checkpoint

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

### Bước 2: Clone mã nguồn từ GitHub & Cài đặt thư viện

In [ ]:
%cd /content
!git clone https://github.com/hoang180/Machine_Translation_Project.git
%cd /content/Machine_Translation_Project
!git checkout feature/p2-experiments-demo

!pip install -q -r requirements.txt
print("✓ Môi trường đã sẵn sàng!")

### Bước 3: Tải dữ liệu IWSLT'15, Tiền xử lý NFC & Huấn luyện 3 bộ Tokenizer (Word, BPE, SPM)

In [ ]:
!bash data/download_iwslt15.sh
!python3 data/preprocess.py

!python3 custom_tokenizers/train_word.py
!python3 custom_tokenizers/train_bpe.py
!python3 custom_tokenizers/train_spm.py

### Bước 4: Huấn luyện các cấu hình thực nghiệm Seq2seq
Chạy lần lượt 4 cấu hình để lấy số liệu so sánh cho báo cáo ACL:
1. **Baseline:** Word + Luong Attention
2. **BPE:** BPE (16k) + Luong Attention
3. **SPM:** SentencePiece (16k) + Luong Attention
4. **Bahdanau:** BPE (16k) + Bahdanau Attention

In [ ]:
# 1. Word + Luong
!python3 train_seq2seq.py --config configs/seq2seq_config.yaml --tokenizer word --attention luong --save_dir checkpoints/seq2seq_word_luong --batch_size 128 --epochs 15 --no_wandb

# 2. BPE + Luong
!python3 train_seq2seq.py --config configs/seq2seq_config.yaml --tokenizer bpe --attention luong --save_dir checkpoints/seq2seq_bpe_luong --batch_size 128 --epochs 15 --no_wandb

# 3. SentencePiece + Luong
!python3 train_seq2seq.py --config configs/seq2seq_config.yaml --tokenizer spm --attention luong --save_dir checkpoints/seq2seq_spm_luong --batch_size 128 --epochs 15 --no_wandb

# 4. BPE + Bahdanau
!python3 train_seq2seq.py --config configs/seq2seq_config.yaml --tokenizer bpe --attention bahdanau --save_dir checkpoints/seq2seq_bpe_bahdanau --batch_size 128 --epochs 15 --no_wandb

### Bước 5: Benchmark Decoding (Greedy vs Beam k=3,5) & Phân tích lỗi theo độ dài câu (`tst2013`)

In [ ]:
# Chạy benchmark decoding + phân tích lỗi theo độ dài (<15, 15-30, >30 từ)
!python3 error_analysis.py --checkpoint checkpoints/seq2seq_bpe_luong/best_model.pt --benchmark_decoding --out_dir outputs/error_analysis_bpe

### Bước 6: Demo dịch thử & Xuất ảnh Attention Heatmap (300 DPI)

In [ ]:
!python3 demo.py --seq2seq_ckpt checkpoints/seq2seq_bpe_luong/best_model.pt --text "When I was little, I thought my country was the best on the planet."

### Bước 7: Sao lưu Checkpoints & Kết quả lên Google Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/BTL_DeepLearning_Translation
!cp -r checkpoints /content/drive/MyDrive/BTL_DeepLearning_Translation/
!cp -r outputs /content/drive/MyDrive/BTL_DeepLearning_Translation/
print("✓ Đã sao lưu toàn bộ checkpoints và bảng kết quả lên Google Drive!")